# Customer360 Retail Analytics
## Production Bronze Ingestion — Category Translation

**Notebook:** `INGEST_CATEGORY_TRANSLATION`

Source: `s3://olist-retail-project/raw/category_translation/`

Target: `workspace.bronze.category_translation`

This notebook only handles incremental Bronze ingestion. The existing
Silver transformation is not changed.

In [0]:
from pyspark.sql import functions as F

In [0]:
# ================================================================
# CONFIGURATION
# ================================================================

SOURCE_PATH = "s3://olist-retail-project/raw/category_translation/"
CHECKPOINT_PATH = (
    "s3://olist-retail-project/_checkpoints/"
    "category_translation_ingestion/"
)
SCHEMA_LOCATION = (
    "s3://olist-retail-project/_schemas/"
    "category_translation_ingestion/"
)

BRONZE_TABLE = "workspace.bronze.category_translation"

EXPECTED_COLUMNS = [
    "product_category_name",
    "product_category_name_english",
]

EXPECTED_TYPES = {
    "product_category_name": "string",
    "product_category_name_english": "string",
}

print("Source     :", SOURCE_PATH)
print("Checkpoint :", CHECKPOINT_PATH)
print("Target     :", BRONZE_TABLE)

Source     : s3://olist-retail-project/raw/category_translation/
Checkpoint : s3://olist-retail-project/_checkpoints/category_translation_ingestion/
Target     : workspace.bronze.category_translation


In [0]:
# ================================================================
# STEP 1 — VERIFY EXISTING BRONZE TARGET
# ================================================================

if not spark.catalog.tableExists(BRONZE_TABLE):
    raise ValueError(
        f"Bronze target does not exist: {BRONZE_TABLE}. "
        "Run the existing Category Translation baseline notebook first."
    )

bronze_before_df = spark.table(BRONZE_TABLE)
before_count = bronze_before_df.count()

print(f"Current Bronze Category Translation rows : {before_count:,}")
print(f"Bronze target                            : {BRONZE_TABLE}")

Current Bronze Category Translation rows : 71
Bronze target                            : workspace.bronze.category_translation


In [0]:
# ================================================================
# STEP 2 — BRONZE SCHEMA CONTRACT
# ================================================================

actual_columns = bronze_before_df.columns

if actual_columns != EXPECTED_COLUMNS:
    raise ValueError(
        "Bronze Category Translation schema contract failed.\n"
        f"Expected columns: {EXPECTED_COLUMNS}\n"
        f"Actual columns  : {actual_columns}"
    )

actual_types = dict(bronze_before_df.dtypes)

type_errors = {
    column: {
        "expected": expected_type,
        "actual": actual_types.get(column),
    }
    for column, expected_type in EXPECTED_TYPES.items()
    if actual_types.get(column) != expected_type
}

if type_errors:
    raise ValueError(
        f"Bronze Category Translation datatype contract failed: {type_errors}"
    )

print("PASS — Existing Bronze schema matches the contract.")
bronze_before_df.printSchema()

PASS — Existing Bronze schema matches the contract.
root
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)



In [0]:
# ================================================================
# STEP 3 — AUTO LOADER CONFIGURATION
# ================================================================
#
# The historical source file is already represented in the existing
# Bronze table. includeExistingFiles=false prevents the first production
# stream start from replaying that historical file.
#
# Future files arriving under the raw/category_translation/ folder are
# processed incrementally.

In [0]:
category_translation_stream_df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.includeExistingFiles", "false")
        .option("cloudFiles.schemaLocation", SCHEMA_LOCATION)
        .option("cloudFiles.inferColumnTypes", "false")
        .option("header", "true")
        .option("mode", "PERMISSIVE")
        .load(SOURCE_PATH)
        .select(
            F.col("product_category_name").cast("string"),
            F.col("product_category_name_english").cast("string"),
        )
)

if category_translation_stream_df.columns != EXPECTED_COLUMNS:
    raise ValueError(
        "Incoming Category Translation schema does not match the Bronze "
        "contract. "
        f"Expected: {EXPECTED_COLUMNS}; "
        f"Actual: {category_translation_stream_df.columns}"
    )

print("PASS — Auto Loader stream configured for new Category Translation files.")
print("PASS — Incoming stream schema matches the Bronze contract.")

PASS — Auto Loader stream configured for new Category Translation files.
PASS — Incoming stream schema matches the Bronze contract.


In [0]:
# ================================================================
# STEP 4 — INCREMENTAL BRONZE APPEND
# ================================================================

query = (
    category_translation_stream_df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", CHECKPOINT_PATH)
        .trigger(availableNow=True)
        .toTable(BRONZE_TABLE)
)

query.awaitTermination()

print("PASS — Category Translation incremental ingestion completed successfully.")

PASS — Category Translation incremental ingestion completed successfully.


In [0]:
# ================================================================
# STEP 5 — POST-INGESTION BRONZE VALIDATION
# ================================================================

bronze_after_df = spark.table(BRONZE_TABLE)
after_count = bronze_after_df.count()

print(f"Current Bronze Category Translation rows : {after_count:,}")
print(f"Bronze target                            : {BRONZE_TABLE}")

if after_count < before_count:
    raise ValueError(
        "Bronze population decreased during incremental ingestion."
    )

print("PASS — Bronze population is non-decreasing.")

Current Bronze Category Translation rows : 71
Bronze target                            : workspace.bronze.category_translation
PASS — Bronze population is non-decreasing.


In [0]:
# ================================================================
# STEP 6 — SCHEMA VALIDATION
# ================================================================

after_columns = bronze_after_df.columns

if after_columns != EXPECTED_COLUMNS:
    raise ValueError(
        "Bronze schema changed after ingestion. "
        f"Expected: {EXPECTED_COLUMNS}; Actual: {after_columns}"
    )

after_types = dict(bronze_after_df.dtypes)

after_type_errors = {
    column: {
        "expected": expected_type,
        "actual": after_types.get(column),
    }
    for column, expected_type in EXPECTED_TYPES.items()
    if after_types.get(column) != expected_type
}

if after_type_errors:
    raise ValueError(
        f"Bronze datatype contract failed after ingestion: {after_type_errors}"
    )

print("PASS — Bronze schema remains unchanged.")
bronze_after_df.printSchema()

PASS — Bronze schema remains unchanged.
root
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)



In [0]:
# ================================================================
# STEP 7 — NULL / BLANK PROFILING
# ================================================================
#
# Bronze preserves source data. These checks report source quality but
# do not silently filter or modify records.

null_profile = bronze_after_df.select(
    F.sum(
        F.when(F.col("product_category_name").isNull(), 1).otherwise(0)
    ).alias("null_product_category_name"),
    F.sum(
        F.when(
            F.col("product_category_name_english").isNull(), 1
        ).otherwise(0)
    ).alias("null_product_category_name_english"),
)

display(null_profile)

blank_profile = bronze_after_df.select(
    F.sum(
        F.when(
            F.col("product_category_name").isNotNull()
            & (F.trim(F.col("product_category_name")) == ""),
            1,
        ).otherwise(0)
    ).alias("blank_product_category_name"),
    F.sum(
        F.when(
            F.col("product_category_name_english").isNotNull()
            & (F.trim(F.col("product_category_name_english")) == ""),
            1,
        ).otherwise(0)
    ).alias("blank_product_category_name_english"),
)

display(blank_profile)

print("PASS — Bronze raw-preservation profiling completed.")

null_product_category_name,null_product_category_name_english
0,0


blank_product_category_name,blank_product_category_name_english
0,0


PASS — Bronze raw-preservation profiling completed.


In [0]:
# ================================================================
# STEP 8 — BUSINESS-KEY DUPLICATE PROFILING
# ================================================================
#
# Silver defines product_category_name as the business key.
# Bronze remains raw and therefore does not deduplicate records.

duplicate_source_categories = (
    bronze_after_df
        .groupBy("product_category_name")
        .count()
        .filter(F.col("count") > 1)
)

duplicate_source_count = duplicate_source_categories.count()

print(
    f"Duplicate source-category groups in Bronze : "
    f"{duplicate_source_count:,}"
)

if duplicate_source_count > 0:
    display(duplicate_source_categories.orderBy(F.desc("count")))
    print(
        "INFO — Bronze preserves source duplicates; Silver owns the "
        "reference-table grain validation."
    )
else:
    print("PASS — No duplicate source-category groups detected.")

Duplicate source-category groups in Bronze : 0
PASS — No duplicate source-category groups detected.


In [0]:
# ================================================================
# STEP 9 — TARGET AVAILABILITY
# ================================================================

if not spark.catalog.tableExists(BRONZE_TABLE):
    raise ValueError(
        f"Bronze target is unavailable after ingestion: {BRONZE_TABLE}"
    )

print("PASS — Bronze Category Translation table is available after ingestion.")

PASS — Bronze Category Translation table is available after ingestion.


In [0]:
# ================================================================
# FINAL SUMMARY
# ================================================================

print("=" * 72)
print("CATEGORY TRANSLATION AUTOMATED INGESTION — SUCCESS")
print("=" * 72)
print(f"Source       : {SOURCE_PATH}")
print(f"Checkpoint   : {CHECKPOINT_PATH}")
print(f"Target       : {BRONZE_TABLE}")
print(f"Rows before  : {before_count:,}")
print(f"Rows after   : {after_count:,}")
print("Mode         : Incremental append")
print("File handling: Auto Loader")
print("Schema mode  : Strict contract validation")
print("Bronze role  : Raw source preservation")
print("=" * 72)

CATEGORY TRANSLATION AUTOMATED INGESTION — SUCCESS
Source       : s3://olist-retail-project/raw/category_translation/
Checkpoint   : s3://olist-retail-project/_checkpoints/category_translation_ingestion/
Target       : workspace.bronze.category_translation
Rows before  : 71
Rows after   : 71
Mode         : Incremental append
File handling: Auto Loader
Schema mode  : Strict contract validation
Bronze role  : Raw source preservation


## Workflow Dependency

`INGEST_CATEGORY_TRANSLATION`

↓

`SILVER_CATEGORY_TRANSLATION`

The existing Silver notebook reads `workspace.bronze.category_translation`
and writes `workspace.silver.category_translation`.

The existing Silver transformation should remain unchanged.